# Necessity vs. Propensity — First Run (3 datasets)

**The question this notebook answers:** do my two task types actually behave differently?

- **Necessity arm — GSM8K** (grade-school math). Reasoning should be *required*: the model
  has to carry numbers across steps it can't do in one shot.
- **Propensity arm — CommonsenseQA & MMLU (non-symbolic)**. Reasoning should be *optional*:
  the answer comes from recall, so writing steps shouldn't help much.

**The test:** run every problem twice — once **with** chain-of-thought, once **without** —
and compare accuracy. A big with/without gap means reasoning was needed (necessity); a
small gap means it wasn't (propensity).

We run **200 problems per dataset**. This is a *first cheap test*: enough to see the
pattern, small enough to finish in one Kaggle GPU session.

**What success looks like:** large gap on GSM8K, small gap on the two propensity sets —
*and* with-CoT accuracy clearly above chance on all of them (a small gap only means
"reasoning optional" if the model can actually do the task).

**Setup:** Kaggle → Settings → Accelerator → GPU (T4 is fine).


## 0. Setup

In [1]:
!pip install -q -U "transformers>=4.51" "accelerate>=0.30" datasets
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 97.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.2 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.

In [1]:
import re
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: True
GPU: Tesla T4


## 1. Config

All the knobs in one place.

- `N_PER_DATASET` — 200, as planned for this first run.
- `MAX_NEW_TOKENS_COT = 2048` — Qwen3 thinking traces are long; less than this truncates
  them and you silently lose accuracy (a bug we already hit once).
- Multiple choice answers are short, so the direct-answer budget can stay small.


In [18]:
MODEL_NAME = "Qwen/Qwen3-1.7B"   # bump to Qwen3-4B if accuracy floors (see final checks)
N_PER_DATASET = 200
MAX_NEW_TOKENS_COT = 1536
MAX_NEW_TOKENS_DIRECT = 32
SEED = 0

torch.manual_seed(SEED)

## 2. Load the model

Qwen3 is a *thinking* model. To get a clean necessity test we control whether it reasons via the `enable_thinking` flag, so the no-CoT condition genuinely has no hidden reasoning.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("Loaded", MODEL_NAME)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-1.7B


## 3. Load the three datasets into one common format

Different datasets, different fields — so we normalise each into the same shape:

```
{"id", "question", "choices" (or None for GSM8K), "gold", "dataset"}
```

- **GSM8K** — free-form numeric answer (no choices). Gold sits after `####`.
- **CommonsenseQA** — 5-way multiple choice; we use the validation split (test has no
  public labels).
- **MMLU non-symbolic** — 4-way multiple choice, filtered to drop math/symbolic questions
  (those behave like necessity, not propensity). We follow Sprague et al.: treat questions
  with an `=` sign or a math subject as symbolic and exclude them here.


In [13]:
LETTERS = ["A", "B", "C", "D", "E"]

# --- GSM8K (necessity) ---
def load_gsm8k(n):
    ds = load_dataset("gsm8k", "main", split="test")
    out = []
    for i in range(min(n, len(ds))):
        sol = ds[i]["answer"]
        m = re.search(r"####\s*([-\d,\.]+)", sol)
        gold = m.group(1).replace(",", "").strip() if m else None
        out.append({"id": f"gsm_{i:04d}", "question": ds[i]["question"],
                    "choices": None, "gold": gold, "dataset": "GSM8K"})
    return out

# --- CommonsenseQA (propensity) ---
def load_csqa(n):
    ds = load_dataset("tau/commonsense_qa", split="validation")
    out = []
    for i in range(min(n, len(ds))):
        ex = ds[i]
        out.append({"id": f"csqa_{i:04d}", "question": ex["question"],
                    "choices": ex["choices"]["text"], "gold": ex["answerKey"],
                    "dataset": "CommonsenseQA"})
    return out

# --- MMLU non-symbolic (propensity) ---
MATH_SUBJECTS = {
    "abstract_algebra", "college_mathematics", "high_school_mathematics",
    "elementary_mathematics", "college_physics", "high_school_physics",
    "high_school_statistics", "college_chemistry", "high_school_chemistry",
    "econometrics", "formal_logic",
}
def is_symbolic(ex):
    if "=" in ex["question"] or any("=" in c for c in ex["choices"]):
        return True
    return ex["subject"] in MATH_SUBJECTS

def load_mmlu(n):
    ds = load_dataset("cais/mmlu", "all", split="test")
    out = []
    for ex in ds:
        if is_symbolic(ex):
            continue
        out.append({"id": f"mmlu_{len(out):04d}", "question": ex["question"],
                    "choices": ex["choices"], "gold": LETTERS[ex["answer"]],
                    "dataset": "MMLU (non-symbolic)"})
        if len(out) >= n:
            break
    return out

problems = load_gsm8k(N_PER_DATASET) + load_csqa(N_PER_DATASET) + load_mmlu(N_PER_DATASET)
print("Loaded:", pd.Series([p["dataset"] for p in problems]).value_counts().to_dict())
print("\nGSM8K example:   ", problems[0]["question"][:80], "| gold:", problems[0]["gold"])
csqa0 = next(p for p in problems if p["dataset"] == "CommonsenseQA")
print("CSQA example:    ", csqa0["question"][:80], "| choices:", csqa0["choices"], "| gold:", csqa0["gold"])
mmlu0 = next(p for p in problems if p["dataset"].startswith("MMLU"))
print("MMLU example:    ", mmlu0["question"][:80], "| gold:", mmlu0["gold"])

Loaded: {'GSM8K': 200, 'CommonsenseQA': 200, 'MMLU (non-symbolic)': 200}

GSM8K example:    Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning an | gold: 18
CSQA example:     A revolving door is convenient for two direction travel, but it also serves as a | choices: ['bank', 'library', 'department store', 'mall', 'new york'] | gold: A
MMLU example:     A lesion causing compression of the facial nerve at the stylomastoid foramen wil | gold: A


## 4. Prompts — same question, two conditions

The only difference between the two conditions is whether the model may reason. Question
and required answer format stay identical, so any accuracy difference can't be blamed on
formatting.

GSM8K asks for a number; the multiple-choice sets ask for a letter. We handle both with
one builder.


In [14]:
def format_question(p):
    if p["choices"] is None:          # GSM8K: free-form
        return p["question"]
    lines = [f"{LETTERS[i]}. {c}" for i, c in enumerate(p["choices"])]
    return p["question"] + "\n" + "\n".join(lines)

def build_prompt(p, use_cot):
    body = format_question(p)
    answer_kind = "X" if p["choices"] is None else "the letter"
    if use_cot:
        instr = (f'Think step by step, then give your final answer on a new line '
                 f'as "The answer is {answer_kind}".')
    else:
        instr = (f'Give ONLY the final answer as "The answer is {answer_kind}". '
                 f'Do not explain. Do not show any work.')
    messages = [{"role": "user", "content": f"{body}\n\n{instr}"}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=use_cot,
    )

print(build_prompt(csqa0, use_cot=True)[:500])

<|im_start|>user
A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?
A. bank
B. library
C. department store
D. mall
E. new york

Think step by step, then give your final answer on a new line as "The answer is the letter".<|im_end|>
<|im_start|>assistant



## 5. Generate + read the answer

Qwen3 writes its reasoning, then `</think>`, then the answer. We only look for the answer *after* `</think>` — otherwise a long or cut-off thinking trace leaks stray numbers/letters into extraction.

In [19]:
# --- Batched generation (the speed fix) ---
# Left-padding is REQUIRED for correct batched decoding on decoder-only models.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Stop tokens: Qwen ends a turn with <|im_end|>. Without this, greedy batched
# generation runs every sequence to the full token cap = very slow.
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = [tokenizer.eos_token_id]
if im_end_id is not None and im_end_id >= 0:
    eos_ids.append(im_end_id)

@torch.no_grad()
def generate_batch(prompts, use_cot):
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024
    ).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_COT if use_cot else MAX_NEW_TOKENS_DIRECT,
        do_sample=False,
        eos_token_id=eos_ids,                 # <-- stop when the turn ends
        pad_token_id=tokenizer.eos_token_id,
    )
    prompt_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(out[i][prompt_len:], skip_special_tokens=True)
            for i in range(len(prompts))]

def answer_part(text):
    return text.split("</think>", 1)[1] if "</think>" in text else text

def extract_number(text):
    ans = answer_part(text)
    m = re.search(r"answer is\s*\$?([-\d,\.]+)", ans, re.IGNORECASE)
    if m:
        return m.group(1).replace(",", "").rstrip(".").strip()
    nums = re.findall(r"-?\d[\d,]*\.?\d*", ans)
    return nums[-1].replace(",", "").rstrip(".").strip() if nums else None

def extract_letter(text):
    ans = answer_part(text)
    # "The answer is C" / "answer is (C)" / "answer is the letter C"
    m = re.search(r"answer is\s*(?:the letter\s*)?\(?\*?\*?([A-E])\b", ans, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # "C)" or "C." as an option marker near the end
    m = re.findall(r"\b([A-E])[\).]", ans)
    if m:
        return m[-1].upper()
    # last resort: a lone capital letter
    letters = re.findall(r"\b([A-E])\b", ans)
    return letters[-1].upper() if letters else None

def is_correct(pred, gold, is_numeric):
    if pred is None or gold is None:
        return False
    if is_numeric:
        try:
            return abs(float(pred) - float(gold)) < 1e-4
        except ValueError:
            return False
    return pred.strip().upper() == gold.strip().upper()

## 6. Diagnostic before the full run

Always check a handful per dataset *before* launching 1,200 generations. We confirm two
things: the CoT condition reaches `</think>` (so reasoning isn't truncated), and the
direct condition has **no** thinking (so the no-CoT test is honest).


In [20]:
for name in ["GSM8K", "CommonsenseQA", "MMLU (non-symbolic)"]:
    p = next(x for x in problems if x["dataset"] == name)
    cot_pred, cot_txt = predict(p, use_cot=True)
    dir_pred, dir_txt = predict(p, use_cot=False)
    print(f"--- {name} | gold={p['gold']} ---")
    print("  CoT reached </think>?", "</think>" in cot_txt, "| pred:", cot_pred)
    print("  DIRECT has thinking? ", "</think>" in dir_txt, "| pred:", dir_pred)
    print("  DIRECT raw:", repr(dir_txt[:100]))
    print()

--- GSM8K | gold=18 ---
  CoT reached </think>? True | pred: 18
  DIRECT has thinking?  False | pred: 16
  DIRECT raw: 'The answer is 16 × 2 = 32.'

--- CommonsenseQA | gold=A ---
  CoT reached </think>? True | pred: D
  DIRECT has thinking?  False | pred: D
  DIRECT raw: 'The answer is D.'

--- MMLU (non-symbolic) | gold=A ---
  CoT reached </think>? True | pred: D
  DIRECT has thinking?  False | pred: C
  DIRECT raw: 'The answer is the letter C.'



**Read the diagnostic before continuing:**
- All three should show `CoT reached </think>? True`. If any is `False`, raise `MAX_NEW_TOKENS_COT`.
- All three should show `DIRECT has thinking? False`. If any is `True`, the no-CoT
  condition is contaminated — stop and check the `enable_thinking` flag.


## 7. The full run

200 problems × 3 datasets × 2 conditions = 1,200 generations. On a T4 this is roughly 30–60 min, dominated by the long CoT generations. Grab a coffee.

In [ ]:
BATCH_SIZE = 8   # raise to 32 if GPU memory is comfortable; drop to 8 if you OOM

def run_condition(problems, use_cot):
    texts = []
    label = "CoT" if use_cot else "direct"
    for i in tqdm(range(0, len(problems), BATCH_SIZE), desc=label):
        batch = problems[i:i + BATCH_SIZE]
        prompts = [build_prompt(p, use_cot) for p in batch]
        texts.extend(generate_batch(prompts, use_cot))
    return texts

# Generate both conditions, batched.
cot_texts = run_condition(problems, use_cot=True)
dir_texts = run_condition(problems, use_cot=False)

records = []
for p, cot_txt, dir_txt in zip(problems, cot_texts, dir_texts):
    is_num = p["choices"] is None
    cot_pred = extract_number(cot_txt) if is_num else extract_letter(cot_txt)
    dir_pred = extract_number(dir_txt) if is_num else extract_letter(dir_txt)
    records.append({
        "id": p["id"], "dataset": p["dataset"], "gold": p["gold"],
        "cot_pred": cot_pred, "cot_correct": is_correct(cot_pred, p["gold"], is_num),
        "direct_pred": dir_pred, "direct_correct": is_correct(dir_pred, p["gold"], is_num),
        "cot_text": cot_txt,   # kept for Stage 3 cue injection
    })

df = pd.DataFrame(records)
print("Done:", len(df), "rows")

CoT:   0%|          | 0/75 [00:00<?, ?it/s]

## 8. The necessity / propensity table

The headline result. The **gap** column is the whole point: large = necessity, small = propensity.

In [ ]:
summary = (
    df.groupby("dataset")
      .agg(n=("id", "count"),
           with_cot_acc=("cot_correct", "mean"),
           no_cot_acc=("direct_correct", "mean"))
)
summary["gap"] = summary["with_cot_acc"] - summary["no_cot_acc"]

# chance accuracy for the "is it above chance?" sanity check
chance = {"GSM8K": 0.0, "CommonsenseQA": 1/5, "MMLU (non-symbolic)": 1/4}
summary["chance"] = summary.index.map(chance)
summary["above_chance"] = summary["with_cot_acc"] > summary["chance"] + 0.05

summary = summary.round(3)
print(summary)
print()
print("Reading it:")
print("  GSM8K  -> expect LARGE gap  (reasoning necessary)")
print("  CSQA / MMLU -> expect SMALL gap (reasoning optional)")
print("  above_chance must be True, or a small gap just means 'can't do the task'.")

## 9. Plot the gap

One simple grouped bar chart: with-CoT vs. no-CoT accuracy per dataset. The size of the gap between the two bars *is* the necessity signal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

order = ["GSM8K", "CommonsenseQA", "MMLU (non-symbolic)"]
s = summary.reindex(order)

x = np.arange(len(order))
w = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, s["with_cot_acc"], w, label="with CoT")
ax.bar(x + w/2, s["no_cot_acc"], w, label="no CoT")

# mark chance level per dataset
for i, d in enumerate(order):
    ax.hlines(chance[d], x[i]-0.45, x[i]+0.45, linestyles="dotted", color="grey")

ax.set_xticks(x)
ax.set_xticklabels(order, rotation=10)
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1)
ax.set_title("Reasoning necessary (big gap) vs. optional (small gap)")
ax.legend()
for i, d in enumerate(order):
    ax.text(x[i], max(s['with_cot_acc'][d], s['no_cot_acc'][d]) + 0.02,
            f"gap {s['gap'][d]:.2f}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("necessity_gap.png", dpi=120)
plt.show()
print("dotted grey line = chance accuracy")

## 10. Save

Keep the per-problem results (including raw CoT text) for Stage 3.

In [ ]:
df.to_csv("stage2_three_datasets.csv", index=False)
print("Saved stage2_three_datasets.csv:", df.shape)
print("Columns:", list(df.columns))

## What to check before trusting this

1. **Did the regimes separate?** GSM8K gap should be large; the two propensity gaps small.
   If GSM8K's gap is small, the model is shortcutting the math — unexpected, investigate.
2. **Is `above_chance` True everywhere?** This is the trap: a 1.7B model can sit near chance
   on multiple-choice in *both* conditions, producing a small gap that looks like propensity
   but is really "can't do it." If with-CoT accuracy isn't clearly above the dotted line,
   rerun that dataset on **Qwen3-4B** before believing the gap.
3. **Did a propensity set show a large gap?** Then it isn't propensity on this model — swap
   it for an easier/more recall-pure set rather than treating it as a puzzle.

If the regimes separate cleanly and everything's above chance, you have two validated arms
and you're ready for **Stage 3: cue injection** — inject a misleading hint into both arms
and measure how often the model takes the bait *without admitting it* in each.
